A transcript assistants that can provides answers to questions from meetings. You only need to provide the transcripts of the meetings. The RAG based product allows you to choose a transcript and ask your questions.

Future improvements would be to load the transcripts from google drive.


In [ ]:
from pathlib import Path
import re
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
from IPython.display import display
import tiktoken


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
INDEX_COLLECTION = "docs_index"  # Maps collection names to original filenames
embedding_model = "text-embedding-3-large"


def filename_to_collection_name(filename: str) -> str:
    """Convert filename to a ChromaDB-safe collection name (3-64 chars, alphanumeric, ., _, -)."""
    sanitized = re.sub(r"[^a-zA-Z0-9._-]", "_", filename)
    name = f"doc_{sanitized}"[:64]
    return name if len(name) >= 3 else f"doc_{filename[:60]}"


def get_indexed_filenames() -> list[str]:
    """Return sorted list of filenames from the index collection."""
    chroma = PersistentClient(path=DB_NAME)
    if INDEX_COLLECTION not in [c.name for c in chroma.list_collections()]:
        return []
    idx = chroma.get_collection(INDEX_COLLECTION)
    result = idx.get(include=["metadatas"])
    filenames = [m["filename"] for m in result["metadatas"] if m and "filename" in m]
    return sorted(set(filenames))
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

# Transcripts folder - local directory containing transcript files (.md, .txt, etc.)
TRANSCRIPTS_FOLDER = Path("transcripts")

openai = OpenAI()

In [ ]:

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")
    keywords: str = Field(description="A list of keywords that are most likely to be surfaced in a query")

    def as_result(self, document):
        metadata = {
            "source": document["source"],
            "type": document["type"],
            "filename": Path(document["source"]).name,
        }
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text + "\n\n" + self.keywords, metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

In [4]:
def fetch_documents():
    """Fetch transcript documents from the local transcripts folder."""

    # Resolve transcripts folder path
    search_dirs = [
        Path.cwd() / TRANSCRIPTS_FOLDER,
        Path.cwd() / "week5" / "community-contributions" / "temmyogunbo" / TRANSCRIPTS_FOLDER,
    ]
    transcripts_path = None
    for path in search_dirs:
        if path.exists():
            transcripts_path = path
            break

    if transcripts_path is None:
        raise FileNotFoundError(
            f"Transcripts folder not found. Tried: {[str(p) for p in search_dirs]}"
        )

    documents = []
    for file in transcripts_path.rglob("*"):
        if file.is_file() and file.suffix.lower() in (".md", ".txt", ".csv", ".json"):
            try:
                with open(file, "r", encoding="utf-8") as f:
                    text = f.read()
                doc_type = file.suffix.lstrip(".") or "document"
                documents.append({
                    "type": doc_type,
                    "source": file.as_posix(),
                    "text": text,
                })
            except Exception as e:
                print(f"Skipped {file.name}: {e}")

    print(f"Loaded {len(documents)} documents from {transcripts_path}")
    return documents

In [5]:
documents = fetch_documents()

Loaded 2 documents from /Users/temmyogunbo/andela/fork/llm_engineering/week5/community-contributions/temmyogunbo/transcripts


In [6]:
display(documents)

[{'type': 'md',
  'source': '/Users/temmyogunbo/andela/fork/llm_engineering/week5/community-contributions/temmyogunbo/transcripts/Expert_Technical Deep Dive - 2026_03_05 15_21 WAT .md',
  'text': 'Mar 5, 2026\n\n## Expert/Technical Deep Dive \\- Transcript\n\n### 00:00:00\n\n\xa0  \n**Ibrahim Kabiru:** See,  \n**MC Elias:** Hi,  \n**Ibrahim Kabiru:** how are you?  \n**MC Elias:** good. How are you doing?  \n**Ibrahim Kabiru:** I\'m fine. How did you know I\'ve joined?  \n**MC Elias:** I didn\'t I also just joined early.  \n**Ibrahim Kabiru:** Okay, let me let me just do it.  \n**MC Elias:** Um  \n**Ibrahim Kabiru:** Oh, I think you I can see you are the meeting host now. I don\'t know just because I added you as a post. I can see the label on your name is say meeting host.  \n**MC Elias:** Say that one more time. What did it say about  \n**Ibrahim Kabiru:** I said I can see uh if you check the the list of uh  \n**MC Elias:** me?  \n**Ibrahim Kabiru:** attendees I can see meeting hosts 

In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a transcript document and you split the transcript document into semantically meaningful overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a user regarding a meeting.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the meetings.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, keywords list, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [10]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [11]:
chunks = create_chunks(documents)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:07<00:00,  4.00s/it]


In [12]:
# Embedding model limit is 8192 tokens; truncate chunks that exceed this
MAX_CHUNK_TOKENS = 8000


def create_embeddings(chunks):
    """Create embeddings for each chunk and store in ChromaDB. One collection per file."""
    enc = tiktoken.get_encoding("cl100k_base")
    chroma = PersistentClient(path=DB_NAME)

    # Group chunks by filename
    by_filename: dict[str, list[tuple[int, Result]]] = {}
    for i, chunk in enumerate(chunks):
        meta = dict(chunk.metadata)
        if "filename" not in meta and "source" in meta:
            meta["filename"] = Path(meta["source"]).name
        filename = meta.get("filename", f"unknown_{i}")
        if filename not in by_filename:
            by_filename[filename] = []
        by_filename[filename].append((i, Result(page_content=chunk.page_content, metadata=meta)))

    # Delete existing collections for these files and the index
    existing = set(c.name for c in chroma.list_collections())
    for filename in by_filename:
        coll_name = filename_to_collection_name(filename)
        if coll_name in existing:
            chroma.delete_collection(coll_name)
    if INDEX_COLLECTION in existing:
        chroma.delete_collection(INDEX_COLLECTION)

    # Create one collection per file
    index_entries = []
    for filename, file_chunks in by_filename.items():
        coll_name = filename_to_collection_name(filename)
        collection = chroma.get_or_create_collection(coll_name)
        for i, (orig_idx, chunk) in enumerate(tqdm(file_chunks, desc=f"Embedding {filename}", leave=False)):
            text = chunk.page_content
            tokens = enc.encode(text)
            if len(tokens) > MAX_CHUNK_TOKENS:
                text = enc.decode(tokens[:MAX_CHUNK_TOKENS])
            emb = openai.embeddings.create(model=embedding_model, input=[text]).data[0]
            collection.add(
                ids=[f"chunk_{orig_idx}"],
                embeddings=[emb.embedding],
                documents=[text],
                metadatas=[chunk.metadata],
            )
        index_entries.append((coll_name, filename))

    # Create index collection (maps collection name -> filename)
    emb_dim = len(openai.embeddings.create(model=embedding_model, input=["x"]).data[0].embedding)
    zero_emb = [0.0] * emb_dim
    idx = chroma.get_or_create_collection(INDEX_COLLECTION)
    for coll_name, filename in index_entries:
        idx.upsert(
            ids=[coll_name],
            embeddings=[zero_emb],
            documents=[filename],
            metadatas=[{"filename": filename}],
        )

    total = sum(len(c) for c in by_filename.values())
    print(f"Vectorstore created: {len(by_filename)} collections, {total} chunks total")

In [13]:
create_embeddings(chunks)

Vectorstore created: 2 collections, 2 chunks total


In [14]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [15]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    print([chunks])
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order

    n = len(chunks)
    # LLM may return 1-based (1..n) or 0-based (0..n-1) indices; filter invalid
    valid_indices = []
    seen = set()
    for i in order:
        # Try 1-based: chunk id 1 -> index 0
        idx_1based = i - 1 if 1 <= i <= n else -1
        # Try 0-based: chunk id 0 -> index 0
        idx_0based = i if 0 <= i < n else -1
        idx = idx_1based if idx_1based >= 0 else idx_0based
        if idx >= 0 and idx not in seen:
            valid_indices.append(idx)
            seen.add(idx)
    # Append any chunks the LLM omitted
    for j in range(n):
        if j not in seen:
            valid_indices.append(j)
    return [chunks[i] for i in valid_indices]

In [16]:
RETRIEVAL_K = 10

def fetch_context_unranked(question, filename_filter=None):
    """Query ChromaDB for relevant chunks. One collection per file; query specific collection when filename provided."""
    chroma = PersistentClient(path=DB_NAME)
    query_emb = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    chunks = []
    print(filename_filter)
    print(question)
    if filename_filter:
        coll_name = filename_to_collection_name(filename_filter)
        collections = chroma.list_collections()
        print(coll_name)
        print(collections)
        if coll_name in [c.name for c in chroma.list_collections()]:
            collection = chroma.get_collection(coll_name)
            results = collection.query(
                query_embeddings=[query_emb],
                n_results=RETRIEVAL_K,
            )
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                chunks.append(Result(page_content=doc, metadata=meta))
    else:
        raise ValueError("No filename filter provided")
    return chunks

In [26]:
question = "What was the purposeof the meeting?"
filename_filter = "Expert_Technical Deep Dive - 2026_03_05 15_21 WAT.md"
chunks = fetch_context_unranked(question, filename_filter)

Expert_Technical Deep Dive - 2026_03_05 15_21 WAT.md
What was the purposeof the meeting?
doc_Expert_Technical_Deep_Dive_-_2026_03_05_15_21_WAT.md
[Collection(name=doc_expert_technical_deep_drive.md), Collection(name=docs_index), Collection(name=doc_Expert_Technical_Deep_Dive_-_2026_03_05_15_21_WAT_.md)]


In [17]:
def fetch_context(question, filename_filter=None):
    """Fetch and rerank context. Use filename_filter to restrict to a specific file by name."""
    chunks = fetch_context_unranked(question, filename_filter)
    return rerank(question, chunks)

In [18]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant that attends meetings.
You are chatting with a user about a meeting.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [19]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [20]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about her meetings.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [21]:
def answer_question(question: str, history: list[dict] = [], filename_filter: str | None = None) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context.
    filename_filter: optional filename to restrict answers to one file (e.g. "transcript.md").
    """
    query = rewrite_query(question, history)
    chunks = fetch_context(query, filename_filter)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
import gradio as gr


# Dropdown lists file names from the index collection
DOCUMENT_CHOICES = get_indexed_filenames()


def chat_turn(question: str, history: list, selected_filename: str) -> tuple[list, str]:
    """Process a question, update chat history, return (history, empty string to clear input)."""
    if not question.strip():
        return history, ""
    # Require a document to be selected
    if not selected_filename:
        feedback = "Please select a document from the dropdown above to search within."
        return history + [(question, feedback)], ""
    # Convert Gradio history [(user, bot), ...] to answer_question format
    msg_history = []
    for user_msg, bot_msg in history:
        msg_history.append({"role": "user", "content": user_msg})
        msg_history.append({"role": "assistant", "content": bot_msg})
    answer, _ = answer_question(question, msg_history, selected_filename)
    return history + [(question, answer)], ""


def clear_on_doc_change():
    """Clear chat history when document selection changes."""
    return []


with gr.Blocks(title="Transcript Q&A") as demo:
    gr.Markdown("# Transcript Assistant - Ask questions about your transcripts")
    with gr.Row():
        with gr.Column(scale=1):
            question_input = gr.Textbox(
                label="Question",
                placeholder="Ask a question about the transcripts...",
                lines=3,
            )
            submit_btn = gr.Button("Submit", variant="primary")
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Chat history", height=400)

    with gr.Row():
        doc_selector = gr.Dropdown(
            choices=DOCUMENT_CHOICES,
            value=DOCUMENT_CHOICES[0] if DOCUMENT_CHOICES else None,
            label="Document",
            info="Select a file to filter context by filename (clears chat when changed)",
        )

    submit_btn.click(
        chat_turn,
        inputs=[question_input, chatbot, doc_selector],
        outputs=[chatbot, question_input],
    )
    question_input.submit(
        chat_turn,
        inputs=[question_input, chatbot, doc_selector],
        outputs=[chatbot, question_input],
    )
    doc_selector.change(clear_on_doc_change, inputs=[doc_selector], outputs=[chatbot])

demo.launch(inbrowser=True)

/var/folders/y9/81b1fwmx5t91jz7g5sc67nnm0000gn/T/ipykernel_12294/4062513083.py:41: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat history", height=400)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Expert_Technical Deep Dive - 2026_03_05 15_21 WAT .md
Who are the participants listed in the meeting?
doc_Expert_Technical_Deep_Dive_-_2026_03_05_15_21_WAT_.md
[Collection(name=doc_Expert_Technical_Deep_Dive_-_2026_03_05_15_21_WAT_.md), Collection(name=docs_index), Collection(name=doc_expert_technical_deep_drive.md)]
[[Result(page_content="Introduction and Meeting Participants\n\nThe meeting begins with participants Ibrahim Kabiru and MC Elias greeting each other and discussing technical aspects of the meeting, including attendee roles and initial setup.\n\nMar 5, 2026\n\n## Expert/Technical Deep Dive - Transcript\n\n### 00:00:00\n\n\xa0  \n**Ibrahim Kabiru:** See,\n**MC Elias:** Hi,\n**Ibrahim Kabiru:** how are you?\n**MC Elias:** good. How are you doing?\n**Ibrahim Kabiru:** I'm fine. How did you know I've joined?\n**MC Elias:** I didn't I also just joined early.\n**Ibrahim Kabiru:** Okay, let me let me just do it.\n**MC Elias:** Um\n**Ibrahim Kabiru:** Oh, I think you I can see you 